In [32]:
import glob
import pandas as pd
import os
from pathlib import Path
import glob
import shutil
import re
import unicodedata

In [33]:
def normalize(text):
    if pd.isnull(text):
        return ""
    # Lowercase
    text = text.lower()
    # Remove accents (e.g., é → e)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    # Strip leading/trailing spaces
    return text.strip().replace(' ', '_')

In [34]:
ROOT = '/Users/tomasandrade/Documents/BSC/ICHOIR/datasets/GTSinger'

In [35]:
lang = 'KO'

In [36]:
output_folder = f'{ROOT}/GTSinger_{lang}_flat'

In [37]:
os.makedirs(f'{output_folder}/wav')
os.makedirs(f'{output_folder}/TextGrid')

In [38]:
all_files = glob.glob(f'{ROOT}/{lang}/*/**/***/****/*****')
data = [file.split(f'{ROOT}/{lang}/')[1].split('/') for file in all_files]

df = pd.DataFrame(data = data, columns=['singer', 'style', 'song', 'group', 'file'])
df['input_path'] = all_files


# prepare output paths
df['song'] = df['song'].apply(normalize)
df['new_song'] = df['style'] + '_' +  df['song'] + '_' + df['group']
df['out_path'] = df['singer'] + '-' + df['new_song'] + '-' + df['file'].str.replace('_TextGrid', '.TextGrid')

In [39]:
df['style'].unique()

array(['Vibrato', 'Pharyngeal', 'Breathy', 'Glissando',
       'Mixed_Voice_and_Falsetto'], dtype=object)

In [43]:
SELECT_STYLES = ['Vibrato', 'Glissando', 'Pharyngeal']
EXCLUDE_GROUPS = ['Paired_Speech_Group', 'Control_Group']

df_songs = df[~df['group'].isin(EXCLUDE_GROUPS)]

df_songs = df_songs[df_songs['style'].isin(SELECT_STYLES)]

df_wav = df_songs[df_songs['file'].str.contains('wav')].sort_values('input_path')
df_wav['out_path'] = f'{ROOT}/GTSinger_{lang}_flat/wav/' + df_wav['out_path']

df_TextGrid = df_songs[df_songs['file'].str.contains('TextGrid')].sort_values('input_path')
df_TextGrid['out_path'] = f'{ROOT}/GTSinger_{lang}_flat/TextGrid/' + df_TextGrid['out_path']

In [44]:
df_wav

,singer,style,song,group,file,input_path,new_song,out_path
971,KO-Soprano-1,Glissando,black_out,Glissando_Group,0000.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_black_out_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
970,KO-Soprano-1,Glissando,black_out,Glissando_Group,0001.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_black_out_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
1049,KO-Soprano-1,Glissando,breathe,Glissando_Group,0000.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_breathe_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
1048,KO-Soprano-1,Glissando,breathe,Glissando_Group,0001.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_breathe_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
1045,KO-Soprano-1,Glissando,breathe,Glissando_Group,0002.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_breathe_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
...,...,...,...,...,...,...,...,...
2467,KO-Tenor-1,Vibrato,,Vibrato_Group,0007.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato__Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
2506,KO-Tenor-1,Vibrato,,Vibrato_Group,0008.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato__Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
2509,KO-Tenor-1,Vibrato,,Vibrato_Group,0009.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato__Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
2462,KO-Tenor-1,Vibrato,,Vibrato_Group,0010.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato__Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...


In [45]:
df_wav.shape, df_TextGrid.shape

((605, 8), (605, 8))

In [46]:
for input_path, out_path in zip(df_wav['input_path'], df_wav['out_path']):
    shutil.copy2(input_path, out_path)

for input_path, out_path in zip(df_TextGrid['input_path'], df_TextGrid['out_path']):
    shutil.copy2(input_path, out_path)

In [47]:
lang

'KO'

# Checks

In [48]:
def check_matching_files(lang):
    tg_root = f'{ROOT}/GTSinger_{lang}_flat/TextGrid'
    wav_root = f'{ROOT}/GTSinger_{lang}_flat/wav'

    tg_files = sorted(glob.glob(f'{tg_root}/*'))
    wav_files = sorted(glob.glob(f'{wav_root}/*'))

    tg_stems = [file.split('/')[-1].split('.TextGrid')[0] for file in tg_files]
    wav_stems = [file.split('/')[-1].split('.wav')[0] for file in wav_files]

    df_tg = pd.DataFrame(data = tg_stems)
    df_wav = pd.DataFrame(data = wav_stems)

    n_wav = len(df_wav)
    n_tg = len(df_tg)

    matches = (df_tg == df_wav).sum().values[0]

    return n_wav, n_tg, matches

In [50]:
check_matching_files('KO')

(216, 216, 216)

In [51]:
from textgrids import TextGrid

In [52]:
def get_all_tg_files(lang):
    tg_root = f'{ROOT}/GTSinger_{lang}_flat/TextGrid'
    tg_files = sorted(glob.glob(f'{tg_root}/*'))

    return tg_files

def get_total_length(lang):
    files = get_all_tg_files(lang)
    secs = sum([get_tg_length(f) for f in files])
    return secs/60.

def get_tg_length(file):
    tg = TextGrid()
    tg.read(file)

    return tg['global'].xmax

In [53]:
get_total_length('KO')

46.941916666666664